<a href="https://colab.research.google.com/github/alfianvx/earthquake-classification-model/blob/main/earthquake_classification_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# STEP 1: IMPORT LIBRARY & INITIALIZATION
# ==========================================
import pandas as pd
import numpy as np
import joblib
import warnings

from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

# Menyembunyikan warning bawaan library agar output rapi
warnings.filterwarnings('ignore', category=UserWarning)
print("Step 1: Semua library berhasil dimuat.")

Step 1: Semua library berhasil dimuat.


In [2]:
# ==========================================
# STEP 2: LOAD DATA & FEATURE ENGINEERING
# ==========================================
# 1. Memuat kolom esensial dari dataset
columns = ['datetime', 'latitude', 'longitude', 'magnitude', 'depth']
df = pd.read_csv('katalog_gempa.tsv', sep='\t', usecols=columns)
df['datetime'] = pd.to_datetime(df['datetime'], format='mixed')
df = df.dropna()

# 2. Rekayasa Fitur Spasial (Membuat Grid Koordinat)
df['lat_grid'] = df['latitude'].round(1)
df['lon_grid'] = df['longitude'].round(1)

# 3. Rekayasa Fitur Temporal (Menghitung selisih waktu antar gempa)
df = df.sort_values(by=['lat_grid', 'lon_grid', 'datetime'])
df['time_diff_hours'] = df.groupby(['lat_grid', 'lon_grid'])['datetime'].diff().dt.total_seconds() / 3600
df['time_diff_hours'] = df['time_diff_hours'].fillna(9999) # Gempa pertama di grid tersebut

print(f"Step 2: Data berhasil dibersihkan. Total baris: {len(df)}")

Step 2: Data berhasil dibersihkan. Total baris: 131134


In [3]:
# ==========================================
# STEP 3: RULE-BASED LABELING UNTUK TARGET KLASIFIKASI (EKSPLISIT)
# ==========================================
df_clf = df.copy()

def label_risk(row):
    # 1. Definisi Aturan Kelas TINGGI
    rule_tinggi = (
        (row['magnitude'] >= 6.5) or
        (row['magnitude'] >= 6.0 and row['depth'] < 50) or
        (row['magnitude'] >= 4.5 and row['time_diff_hours'] <= 24)
    )

    # 2. Definisi Aturan Kelas SEDANG
    # (Magnitudo >= 5.0 DAN tidak memenuhi aturan Tinggi)
    rule_sedang = (row['magnitude'] >= 5.0) and not rule_tinggi

    # 3. Eksekusi Pelabelan
    if rule_tinggi:
        return 'Tinggi'
    elif rule_sedang:
        return 'Sedang'
    else:
        # Aturan Kelas RENDAH (Selain Tinggi dan Sedang)
        return 'Rendah'

# Menerapkan label ke dataset
df_clf['risk_category'] = df_clf.apply(label_risk, axis=1)

# Mengubah label teks menjadi angka (0, 1, 2)
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df_clf['target_class'] = encoder.fit_transform(df_clf['risk_category'])

print("Step 3: Pelabelan target risiko (Eksplisit) berhasil dilakukan.")

Step 3: Pelabelan target risiko (Eksplisit) berhasil dilakukan.


In [4]:
# ==========================================
# STEP 4 & 5: DATA SPLIT & PIPELINE KLASIFIKASI (DENGAN SCALER)
# ==========================================
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler # <--- IMPORT SCALER
import pandas as pd

# 1. Mengurutkan ulang secara kronologis
df_clf = df_clf.sort_values(by='datetime')

X_c = df_clf[['magnitude', 'depth', 'latitude', 'longitude', 'time_diff_hours']]
y_c = df_clf['target_class']

# 2. Membagi data (80% Latih, 20% Uji) TANPA Shuffle
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, shuffle=False)

# 3. MEMBANGUN PIPELINE DENGAN SCALER
pipeline_clf = ImbPipeline([
    ('scaler', StandardScaler()), # <--- LANGKAH 1: Setarakan skala semua fitur (Magnitude vs Jam)
    ('smote', SMOTE(random_state=42)), # <--- LANGKAH 2: Buat data sintetis secara adil
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)) # <--- LANGKAH 3: Latih AI
])

# 4. Melatih Pipeline
print("Melatih Pipeline Klasifikasi (Scaler -> SMOTE -> XGBoost)...")
pipeline_clf.fit(X_train_c, y_train_c)

# 5. Evaluasi ke data uji
y_pred_c = pipeline_clf.predict(X_test_c)

# Menampilkan Hasil Evaluasi
print(f"\n--- EVALUASI PIPELINE KLASIFIKASI ---")
print(f"Akurasi Keseluruhan : {accuracy_score(y_test_c, y_pred_c):.4f}\n")

nama_kelas = encoder.inverse_transform([0, 1, 2])
print("Classification Report:")
print(classification_report(y_test_c, y_pred_c, target_names=nama_kelas))

print("Confusion Matrix:")
cm = confusion_matrix(y_test_c, y_pred_c)
df_cm = pd.DataFrame(cm, index=[f"Aktual {k}" for k in nama_kelas], columns=[f"Prediksi {k}" for k in nama_kelas])
print(df_cm)

Melatih Pipeline Klasifikasi (Scaler -> SMOTE -> XGBoost)...

--- EVALUASI PIPELINE KLASIFIKASI ---
Akurasi Keseluruhan : 0.9997

Classification Report:
              precision    recall  f1-score   support

      Rendah       1.00      1.00      1.00     25795
      Sedang       1.00      0.99      0.99       286
      Tinggi       0.98      0.99      0.98       146

    accuracy                           1.00     26227
   macro avg       0.99      0.99      0.99     26227
weighted avg       1.00      1.00      1.00     26227

Confusion Matrix:
               Prediksi Rendah  Prediksi Sedang  Prediksi Tinggi
Aktual Rendah            25793                1                1
Aktual Sedang                1              283                2
Aktual Tinggi                2                0              144


In [5]:
# ==========================================
# STEP 6: EKSTRAKSI TARGET ESTIMASI KEWASPADAAN
# ==========================================
df_reg = df.copy()
event_limit = 72 # Batas 72 jam untuk mendefinisikan siklus gempa baru

# Mendeteksi siklus kejadian (Event)
df_reg['is_new_event'] = (df_reg['time_diff_hours'] > event_limit)
df_reg['event_id'] = df_reg.groupby(['lat_grid', 'lon_grid'])['is_new_event'].cumsum()

# Menghitung durasi historis dari gempa utama hingga gempa susulan terakhir mereda
event_stats = df_reg.groupby(['lat_grid', 'lon_grid', 'event_id'])['datetime'].agg(['min', 'max']).reset_index()

# Mengganti nama kolom agar sesuai dengan konsep kewaspadaan mitigasi
event_stats['durasi_kewaspadaan_jam'] = (event_stats['max'] - event_stats['min']).dt.total_seconds() / 3600

# Memasukkan target ke data Gempa Utama (Magnitudo >= 4.5)
df_main = df_reg[df_reg['is_new_event'] == True].copy()
df_main = pd.merge(df_main, event_stats[['lat_grid', 'lon_grid', 'event_id', 'durasi_kewaspadaan_jam']],
                   on=['lat_grid', 'lon_grid', 'event_id'], how='left')

df_reg_final = df_main[df_main['magnitude'] >= 4.5].copy()

print("Step 6: Data historis durasi kewaspadaan berhasil diekstrak.")

Step 6: Data historis durasi kewaspadaan berhasil diekstrak.


In [6]:
# ==========================================
# STEP 7: TRAINING & EVALUASI ESTIMASI KEWASPADAAN
# ==========================================
# Mengurutkan ulang secara kronologis
df_reg_final = df_reg_final.sort_values(by='datetime')

X_r = df_reg_final[['magnitude', 'depth', 'latitude', 'longitude']]

# [REVISI] Memanggil nama kolom yang baru sesuai Step 6
y_r = df_reg_final['durasi_kewaspadaan_jam']

# Split Data (Tanpa Shuffle)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_r, y_r, test_size=0.2, shuffle=False)

# Latih Regressor
model_reg = XGBRegressor(objective='reg:squarederror', n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42)
model_reg.fit(X_train_r, y_train_r)

# Evaluasi Error Absolut (Tidak boleh prediksi jam minus)
y_pred_r = np.maximum(0, model_reg.predict(X_test_r))
mae = mean_absolute_error(y_test_r, y_pred_r)

# [REVISI] Menggunakan terminologi baru untuk hasil print
print(f"Step 7: Model Estimasi Kewaspadaan Selesai. Margin Error (MAE): {mae:.2f} Jam")

Step 7: Model Estimasi Kewaspadaan Selesai. Margin Error (MAE): 5.41 Jam


In [7]:
# ==========================================
# STEP 8: EXPORT MODEL ARTIFACTS
# ==========================================

# [REVISI] Menyimpan pipeline utuh, BUKAN model_clf biasa
joblib.dump(pipeline_clf, 'pipeline_xgboost_gempa.pkl')

joblib.dump(encoder, 'label_encoder.pkl')
joblib.dump(model_reg, 'xgboost_cooldown.pkl')

print("Step 8: Semua model berhasil disimpan menjadi file .pkl! Siap di-deploy ke Backend.")

Step 8: Semua model berhasil disimpan menjadi file .pkl! Siap di-deploy ke Backend.
